In [ ]:
import ast
from time import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


from PySparseCoalescedTsetlinMachineCUDA.tm import MultiOutputTsetlinMachine

In [ ]:
filepath = '../../data/stortinget_sammendrag.csv.gz'
df = pd.read_csv(filepath)
df['emneord'] = df['emneord'].apply(ast.literal_eval)
df.head()

df = df.dropna(subset=['sammendrag'])

print("\n\n".join(df['sammendrag'].astype(str).tolist()))

In [ ]:
all_emneord = np.sort(np.unique(df['emneord'].sum()))
emnedict = {str(k): v for v, k in zip(range(len(all_emneord)), all_emneord)}

print(f"Number of unique labels: {len(emnedict)}")

tags, counts = np.unique(df['emneord'].sum(), return_counts=True)
common_tags_ind = np.argsort(-counts)[:160]

print("Most common labels\n", "-"*20)
emnedict_reduced = {}
for i, tag in enumerate(common_tags_ind):
    print(f"{tags[tag]} -- {counts[tag]}")
    emnedict_reduced[str(tags[tag])] = i

print(f"\nReduced emnedict: {emnedict_reduced}")

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split as tts
import advertools as adv
import spacy
from tqdm import tqdm

stopwords = [ 'og', 'til', 'av', 'for', 'om', 'fra', 'dette', 'ordf', 'st', 'jf',
                'at', 'på', 'det', 'med', 'som', 'en', 'er', 'den', 'kap',
                'har', 'frå', 'de', 'skal', 'nr', 'også', 'kan', 'et', 'nå',
                'ei', 'merknader', 'komiteens', 'tilrading', 'tilråding', 'eller',
                'innst', 'kr', 'mill', 'mrd', 'kroner', 'pdf', 'neste', 'side', 'publikasjonen',
                'ordfører', 'leder', 'alle', 'prop', 'innhold', 'søk', 'toppen', 
                'kapittel', 'sidetall', 'alt', 'tone', 'bakgrunn', 'vedlegg', 'sammendrag', 'hele', 'vedtak', 'under',
                'forslag', 'gå', 'vis', 'kildedok', 'klikk', 'saker', 'sak', 'dokumentet', 'innstilling', 'innhold',
                'dokument', 'alstahaug', 'alta', 'alvdal', 'alver', 'andøy', 'aremark', 'arendal', 'asker', 'askvoll', 
                'askøy', 'aure', 'aurland', 'aurskog-høland', 'austevoll', 'austrheim', 'averøy', 'balsfjord', 'bamble', 
                'bardu', 'beiarn', 'berg', 'bergen', 'berlevåg', 'bindal', 'birkenes', 'bjarkøy', 'bjerkreim', 'bjørnafjorden', 
                'bø', 'bodø', 'bokn', 'bremanger', 'brønnøy', 'bygland', 'bykle', 'bærum', 'dovre', 'drammen', 'drangedal', 'dyrøy', 
                'dønna', 'eidfjord', 'eidskog', 'eidsvoll', 'eigersund', 'elverum', 'enebakk', 'engerdal', 'etne', 'etnedal', 'evenes', 
                'evje og hornnes', 'farsund', 'fauske', 'fedje', 'fitjar', 'fjaler', 'fjell', 'flakstad', 'flatanger', 'flekkefjord', 
                'flesberg', 'flå', 'folldal', 'forsand', 'fosen', 'fosnes', 'frederikstad', 'froland', 'frosta', 'frøya', 'fyresdal', 
                'gamvik', 'gaular', 'gausdal', 'gildeskål', 'giske', 'gjemnes', 'gjerdrum', 'gjerstad', 'gjesdal', 'gjøvik', 'gloppen', 
                'gol', 'gran', 'grane', 'gratangen', 'grimstad', 'grong', 'grue', 'gulen', 'hadsel', 'halden', 'halsa', 'hamar', 'hamarøy', 
                'hammerfest', 'haram', 'hareid', 'harstad', 'hasvik', 'hattfjelldal', 'haugesund', 'hemnes', 'hemne', 'hemsedal', 'hitra', 
                'hjartdal', 'hjelmeland', 'hobøl', 'hol', 'hole', 'holmestrand', 'holtålen', 'hornindal', 'horten', 'hurdal', 'hurum', 
                'hvaler', 'hyllestad', 'høyanger', 'høylandet', 'inderøy', 'inderøy kommune', 'iveland', 'jevnaker', 'karasjok', 'karlshus', 
                'karmøy', 'kautokeino', 'klepp', 'klæbu', 'kommune', 'kragerø', 'kristiansand', 'kristiansund', 'krødsherad', 'kvalsund', 
                'kvam', 'kvinesdal', 'kvinnherad', 'kviteseid', 'kvitsøy', 'kåfjord', 'larvik', 'lavangen', 'lebesby', 'leikanger', 'leirfjord', 
                'leka', 'leksdal', 'lendal', 'lesja', 'levanger', 'lier', 'lierne', 'lillehammer', 'lillesand', 'lindesnes', 'lindås', 'lom', 
                'loppa', 'lund', 'lunner', 'luroy', 'luster', 'lyngdal', 'lyngen', 'lødingen', 'malvik', 'mandal', 'marker', 'marnardal', 
                'masfjorden', 'meland', 'meldal', 'melhus', 'meløy', 'meråker', 'mesna', 'midtre gauldal', 'modalen', 'modum', 'mo i rana', 
                'moelv', 'moskenes', 'moss', 'mosvik', 'muladal', 'måsøy', 'namdalseid', 'namsskogan', 'namsos', 'nannestad', 'narvik', 'naustdal', 
                'nes', 'nesbyen', 'nesna', 'nesodden', 'nesset', 'nissedal', 'nittedal', 'nord-aurdal', 'norddal', 'nord-fron', 'nordreisa', 
                'nore og uvdal', 'notodden', 'nærøy', 'oppdal', 'oppegård', 'orkdal', 'orkland', 'os', 'oslo', 'osterøy', 'overhalla', 'ovre eiker', 
                'øksnes', 'østre toten', 'øvre eiker', 'øyer', 'øygarden', 'øystre slidre', 'porsanger', 'porsgrunn', 'radøy', 'radoy', 'rauma', 
                'rendalen', 'rennebu', 'rindal', 'ringebu', 'ringerike', 'ringsaker', 'risør', 'roan', 'rollag', 'roros', 'røst', 'rydberg',
                'røyken', 'røros', 'røyrvik', 'råde', 'råde', 'rømskog', 'salangen', 'saltdal', 'samnanger', 'sande', 'sandefjord', 'sandnes', 
                'sanngfjord', 'sarpsborg', 'sauda', 'sel', 'selbu', 'selje', 'seljord', 'sigdal', 'siljan', 'sirdal', 'skaun', 'skedsmo', 'ski', 
                'skien', 'skiptvet', 'skjervøy', 'skjåk', 'smøla', 'snillfjord', 'snåsa', 'snåasen', 'sogndal', 'sokndal', 'sola', 'solund', 
                'songdalen', 'sortland', 'spitsbergen', 'spydeberg', 'stange', 'stavanger', 'stavern', 'stedje', 'steigen', 'steinkjer', 'stjørdal', 
                'stokke', 'stor-elvdal', 'stord', 'stordal', 'storjord', 'stranda', 'strand', 'stryn', 'sula', 'suldal', 'sund', 'sunndal', 'surnadal', 
                'sveio', 'sykkylven', 'søgne', 'søndre land', 'sør-aurdal', 'sør-fron', 'sørfold', 'sørreisa', 'sørum', 'sør-varanger', 'tana', 'time', 
                'tingvoll', 'tinn', 'tjeldsund', 'tjome', 'tokke', 'tolga', 'torsken', 'tranøy', 'trondheim', 'tvedestrand', 'tydal', 'tynset', 'tysnes', 
                'tysnes', 'tysvaer', 'tysvær', 'tønsberg', 'ullensaker', 'ullensvang', 'ulvik', 'utsira', 'vadso', 'vadsø', 'vaga', 'vaksdal', 'valle', 
                'vang', 'vanylven', 'vardo', 'vardø', 'vargje', 'vefsn', 'vega', 'vegårshei', 'vennesla', 'verdal', 'verøy', 'vestby', 'vestnes', 
                'vestre slidre', 'vestre toten', 'vestvågøy', 'vevelstad', 'vik', 'vikna', 'vindafjord', 'vinje', 'volda', 'voss', 'åfjord', 'ål',
                'ålesund', 'åmli', 'åmot', 'årdal', 'ås', 'åsnes', 'januar', 'februar', 'mars', 'april', 'mai', 'juni', 'juli', 'august', 'september', 
                'oktober', 'november', 'desember', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '00', '01', '02', '03', '04', '05', '06', '07', '08', 
                '09', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21', 
                '22', '23', '24', '25', '26', '27', '28', '29', '30', '31', '32', '33', '34', '35', '36', '37', '38', '39', '40', '41', '42', '43', 
                '44', '45', '46', '47', '48', '49', '50', '51', '52', '53', '54', '55', '56', '57', '58', '59', '60', '61', '62', '63', '64', '65', 
                '66', '67', '68', '69', '70', '71', '72', '73', '74', '75', '76', '77', '78', '79', '80', '81', '82', '83', '84', '85', '86', '87', 
                '88', '89', '90', '91', '92', '93', '94', '95', '96', '97', '98', '99', '100', '000', '1995', '1996', '1997', '1998', '1999', '2000', '2001', '2002', '2003', '2004', '2005', '2006', '2007', '2008', '2009', 
                '2010', '2011', '2012', '2013', '2014', '2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024', '2025', '2026'
                ]

adv_stopwords = adv.stopwords['norwegian']

combined_stopwords = set(stopwords) | set(adv_stopwords)

with open("stopwords.txt", "w", encoding="utf-8") as f:
    for word in sorted(combined_stopwords):
        f.write(word + "\n")


train, test = tts(df, test_size=0.15, random_state=1)
test_samples = test.index
train = train.reset_index(drop=True)
test = test.reset_index(drop=True)

corpus = list(train.loc[:, 'sammendrag'].values)
vect = CountVectorizer(
    ngram_range=(1, 3),
    max_features=10000,
    binary=True,
    stop_words=list(combined_stopwords)
)

tic = time()
vect.fit(corpus)
toc = time()
print(f"Vectorizer trained in {toc - tic:.3f} seconds")

tic = time()
X_train = vect.transform(corpus).toarray()
toc = time()
print(f"Corpus transformed in {toc-tic:.3f} seconds")

tic = time()
X_test = vect.transform(list(test.loc[:, 'sammendrag'].values)).toarray()
toc = time()
print(f"Test data transformed in {toc-tic:.3f} seconds")

In [ ]:
# Get the encoded labels

emneset = set(emnedict_reduced.keys())
def get_target_vect(df, emneset):
    Y = np.zeros([df.shape[0], len(emneset)], dtype=int)
    for i in range(df.shape[0]):
        labels = set(df.loc[i, 'emneord']).intersection(emneset)
        for l in labels:
            Y[i, emnedict_reduced[l]] = 1
    return Y


Y_train = get_target_vect(train.iloc[:], emneset)
Y_test = get_target_vect(test.iloc[:], emneset)

# Remove samples without any labels
X_train = X_train[np.sum(Y_train, axis=1) > 0]
Y_train = Y_train[np.sum(Y_train, axis=1) > 0]
X_test = X_test[np.sum(Y_test, axis=1) > 0]
Y_test = Y_test[np.sum(Y_test, axis=1) > 0]

print(f"Number of training samples: {X_train.shape[0]}")
print(f"Number of test samples:     {X_test.shape[0]}")

In [ ]:
from sklearn.metrics import classification_report
import os

tm = MultiOutputTsetlinMachine(
    number_of_clauses=20000,
    T=41000,
    s=5.0,
    q=7,
    max_included_literals=32,
)

SEED_ID = 1

results_file = "classification_reports_countvectorizer_sammendrag_160_emneord.csv"
results = []

for i in range(50):
    tm.fit(X_train, Y_train, epochs=1, incremental=True), 
    Y_pred, cs = tm.predict(X_test)

    print(f"-- Epoch {i} --", end='\n')
    print(classification_report(Y_test, Y_pred, target_names=emnedict_reduced.keys(), zero_division=np.nan))

final_pred, cs = tm.predict(X_test)

final_report_dict = classification_report(
    Y_test, final_pred,
    target_names=list(emnedict_reduced.keys()),
    zero_division=np.nan,
    output_dict=True
)


final_df = pd.DataFrame(final_report_dict).T
final_df["run"] = SEED_ID
final_df["epoch"] = i 

if not os.path.exists(results_file):
    final_df.to_csv(results_file, index=True)
else:
    final_df.to_csv(results_file, mode="a", header=False, index=True)